<a href="https://colab.research.google.com/github/utpalssg/pythonai/blob/newBranch/TensorTry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from getpass import getpass
token = getpass('Enter your GitHub token:')
!git clone --recurse-submodules https://{token}@github.com/utpalssg/pythonai.git

Enter your GitHub token:··········
Cloning into 'pythonai'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 96 (delta 47), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 1.85 MiB | 5.26 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [3]:
import torch
from torchvision import models, transforms

# using cpu
#model = models.resnet50(pretrained=True)

# using gpu
model = models.resnet50(pretrained=True).to("cuda")

  warnings.warn(

  warnings.warn(msg)



In [4]:
from PIL import Image

img = Image.open("/content/pythonai/inference/img1.jpg")


In [5]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
# Apply the transform to the original PIL image from cell cOmxknVSJv8t
img_tensor = transform(img)
img_tensor.shape

torch.Size([3, 224, 224])

In [6]:
import torch

img_batch = torch.unsqueeze(img_tensor, 0).to("cuda")
img_batch.shape

torch.Size([1, 3, 224, 224])

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)             # Move model to GPU
img_batch = img_batch.to(device)     # Move input to GPU

model.eval()
with torch.no_grad():
    outputs = model(img_batch)
probs = torch.nn.functional.softmax(outputs[0], dim=0)

In [8]:
import pandas as pd

labels = pd.read_csv("https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt", header=None)
labels[0][3]

'tiger shark'

In [9]:
topk=5
prob, class_number = torch.topk(probs, topk)
for i in range(topk):
    probability = prob[i].item()
    class_name = labels[0][int(class_number[i])]
    print(f"{class_name}: {probability * 100:.2f}%")

Egyptian cat: 37.12%
tabby: 19.35%
tiger cat: 11.50%
Siamese cat: 4.54%
carton: 3.44%


In [10]:
import numpy as np
import time
import torch.backends.cudnn as cudnn
cudnn.benchmark = True

def rn50_preprocess():
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return preprocess

# decode the results into ([predicted class, description], probability)
def predict(img_path, model):
    img = Image.open(img_path)
    preprocess = rn50_preprocess()
    input_tensor = preprocess(img)
    input_batch = input_tensor.unsqueeze(0) # create a mini-batch as expected by the model

    # move the input and model to GPU for speed if available
    if torch.cuda.is_available():
        input_batch = input_batch.to('cuda')
        model.to('cuda')

    with torch.no_grad():
        output = model(input_batch)
        # Tensor of shape 1000, with confidence scores over Imagenet's 1000 classes
        sm_output = torch.nn.functional.softmax(output[0], dim=0)

    ind = torch.argmax(sm_output)
    return d[str(ind.item())], sm_output[ind] #([predicted class, description], probability)

def benchmark(model, device="cuda", input_shape=(1, 3, 224, 224), dtype='fp32', nwarmup=50, nruns=100):
    input_data = torch.randn(input_shape)
    input_data = input_data.to(device)
    #if dtype=='fp16':
    #    input_data = input_data.half()

    print("Warm up ...")
    with torch.no_grad():
        for _ in range(nwarmup):
            features = model(input_data)
    torch.cuda.synchronize()
    print("Start timing ...")
    timings = []
    with torch.no_grad():
        for i in range(1, nruns+1):
            start_time = time.time()
            features = model(input_data)
            torch.cuda.synchronize()
            end_time = time.time()
            timings.append(end_time - start_time)
            if i%10==0:
                print('Iteration %d/%d, ave batch time %.2f ms'%(i, nruns, np.mean(timings)*1000))

    print("Input shape:", input_data.size())
    print("Output features size:", features.size())
    print('Average batch time: %.2f ms'%(np.mean(timings)*1000))

In [11]:
benchmark(model, device="cuda")

Warm up ...
Start timing ...
Iteration 10/100, ave batch time 6.53 ms
Iteration 20/100, ave batch time 6.38 ms
Iteration 30/100, ave batch time 7.14 ms
Iteration 40/100, ave batch time 6.96 ms
Iteration 50/100, ave batch time 6.84 ms
Iteration 60/100, ave batch time 6.78 ms
Iteration 70/100, ave batch time 6.76 ms
Iteration 80/100, ave batch time 6.71 ms
Iteration 90/100, ave batch time 6.68 ms
Iteration 100/100, ave batch time 6.65 ms
Input shape: torch.Size([1, 3, 224, 224])
Output features size: torch.Size([1, 1000])
Average batch time: 6.65 ms


In [12]:
import torch
traced_model=torch.jit.trace(model, [torch.randn((1,3,224,224)).to('cuda')])

In [ ]:
pip install torch-tensorrt -f https://download.pytorch.org/whl/torch_tensorrt_nightly.html

In [ ]:
pip install --upgrade torchvision torchaudio fastai

In [13]:
import torch_tensorrt

trt_model=torch_tensorrt.compile(traced_model,
                                 ir="torchscript",
                                 inputs=[torch_tensorrt.Input((1,3,224,224), dtype=torch.float32)],
                                 enabled_precisions={torch.float32}
                                 )

In [14]:
benchmark(trt_model, device="cuda")

Warm up ...
Start timing ...
Iteration 10/100, ave batch time 4.02 ms
Iteration 20/100, ave batch time 4.01 ms
Iteration 30/100, ave batch time 3.83 ms
Iteration 40/100, ave batch time 3.71 ms
Iteration 50/100, ave batch time 3.63 ms
Iteration 60/100, ave batch time 3.58 ms
Iteration 70/100, ave batch time 3.55 ms
Iteration 80/100, ave batch time 3.53 ms
Iteration 90/100, ave batch time 3.50 ms
Iteration 100/100, ave batch time 3.49 ms
Input shape: torch.Size([1, 3, 224, 224])
Output features size: torch.Size([1, 1000])
Average batch time: 3.49 ms


In [15]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# trt_model = trt_model.to(device)     # Move model to GPU
# img_batch = img_batch.to(device)     # Move input to GPU

trt_model.eval()
with torch.no_grad():
    outputs = trt_model(img_batch)
probs = torch.nn.functional.softmax(outputs[0], dim=0)

topk=5
prob, class_number = torch.topk(probs, topk)
for i in range(topk):
    probability = prob[i].item()
    class_name = labels[0][int(class_number[i])]
    print(f"{class_name}: {probability * 100:.2f}%")

Egyptian cat: 37.12%
tabby: 19.35%
tiger cat: 11.50%
Siamese cat: 4.54%
carton: 3.44%
